In [15]:
import os

In [16]:
pwd

'/Users/shanmukhamundra/PycharmProjects/NLP-Text-Summarization'

In [17]:
os.chdir('/Users/shanmukhamundra/PycharmProjects/NLP-Text-Summarization')

In [18]:
pwd

'/Users/shanmukhamundra/PycharmProjects/NLP-Text-Summarization'

In [19]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataTransformationConfig:
    tokenizer_name: str
    data_path: Path
    root_dir: Path


In [20]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [21]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])



    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

In [22]:
import os
from transformers import PegasusTokenizer
from datasets import load_from_disk

In [23]:
class DataTransformation:
    def __init__(self, config):
        self.config = config
        try:
            self.tokenizer = PegasusTokenizer.from_pretrained(config.tokenizer_name)
        except Exception as e:
            raise ValueError(f"Failed to load PegasusTokenizer: {str(e)}")

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True, padding="max_length")
        with self.tokenizer.as_target_tokenizer():
            target_encodings = self.tokenizer(example_batch['summary'], max_length=128, truncation=True, padding="max_length")
        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir, "samsum_dataset"))


In [24]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2025-06-10 19:10:17,837: INFO: common: yaml file: config/config.yaml loaded successfully]
[2025-06-10 19:10:17,841: INFO: common: yaml file: params.yaml loaded successfully]
[2025-06-10 19:10:17,842: INFO: common: created directory at: artifacts]
[2025-06-10 19:10:17,844: INFO: common: created directory at: artifacts/data_transformation]


Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 75392.03 examples/s]
